In [ ]:
import os
import csv
import pandas as pd
import multiprocessing

sample_size = 1
min_distance = 2
random_seed = 4
network_type = 'drive'
point_distance_size = 10000
experiment_name = "2025-07-evaluation"
base_path=f"/home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/{experiment_name}"
print(base_path)
min_od_distance = 4750
max_od_distance = 5250
od_pair_sample_size = 3


if not os.path.exists(base_path):
    os.makedirs(base_path)

parameters_file_path = os.path.join(base_path, f"parameters.csv")
city_sample_nodes_path = os.path.join(base_path, f'city_sample_nodes_{experiment_name}.csv')
local_graph_folder = os.path.join(base_path, 'local_origin_graphs')

with open(parameters_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["Parameter", "Value"])
    writer.writerow(["sample_size", sample_size])
    writer.writerow(["min_distance", min_distance])
    writer.writerow(["random_seed", random_seed])
    writer.writerow(["network_type", network_type])
    writer.writerow(["point_distance_size", point_distance_size])
    writer.writerow(["min_od_distance", min_od_distance])
    writer.writerow(["max_od_distance", max_od_distance])
    writer.writerow(["base_path", base_path])
    writer.writerow(["city_sample_nodes_path", city_sample_nodes_path])
    writer.writerow(["local_graph_folder", local_graph_folder])
    writer.writerow(["base_path", base_path])
    writer.writerow(["experiment_name", experiment_name])


param = pd.read_csv(parameters_file_path)
display(param)



num_processes = multiprocessing.cpu_count()  # Adjust based on your system's capabilities
print(f"Number of processes to use: {num_processes}")
print(city_sample_nodes_path)

In [ ]:
import pandas as pd # for reading the csv file
import joblib # replacing multiprocessing with joblib
import route_network_analysis as rna
import os
import csv
import glob
min_od_distance = 4750
max_od_distance = 5250
od_pair_sample_size = 36
experiment_name = "2025-07-evaluation"
base_path=f"/home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/{experiment_name}"
city_sample_nodes_path = os.path.join(base_path, f'city_sample_nodes_{experiment_name}.csv')
df = pd.read_csv(city_sample_nodes_path)
display(df)
local_odpair_folder = os.path.join(base_path, "od_pair_data")
local_odpair_base_folder = os.path.join(local_odpair_folder, 'base')
local_odpair_geom_folder = os.path.join(local_odpair_folder, 'geom')
print(f"odpair data will be stored at {local_odpair_folder}")
os.makedirs(local_odpair_folder, exist_ok=True)
os.makedirs(local_odpair_base_folder, exist_ok=True)
os.makedirs(local_odpair_geom_folder, exist_ok=True)

plotpath = "evaluation/2025-07-evaluation/plots"
df['od_pairs_added'] = False


def get_od_pairs(row):
    print(f"Finding OD_pairs for graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    og = rna.origin_graph.from_graphml(graphml_path=row['graph_path'])
    og.create_od_pairs(min_radius=min_od_distance, max_radius=max_od_distance, sample_size=od_pair_sample_size)
    n = 1
    node_ids = []
    for od_pair in og.od_pairs:
        if n % 4 == 0:
            print(n)
            node_ids.append(od_pair.id)
            graph_plot_path = os.path.join(plotpath,f"{od_pair.id}_graph.png")
            html_path = os.path.join(plotpath,f"{od_pair.id}.html")
            od_pair.plot_on_map(html_path,graph_plot_path)

        n=n+1

    od_pair_data = og.get_od_pair_data()
    mask = od_pair_data['id'].isin(node_ids)
    od_pair_data = od_pair_data[mask]
    csv_base_path = os.path.join(local_odpair_base_folder, f"od_pair_{row['city_name_en']}_{row['node_id']}.csv")
    od_pair_data.to_csv(csv_base_path)

    print(f"Finished finding OD_pairs for graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    return True,row['city_name_en'],row['node_id']


for idx, row in df.iterrows():
    try:
        get_od_pairs(row)
    except Exception as e:
        print(f"error {e}")

# Get all json files from od_pair_data folder
od_pair_base_files = glob.glob(os.path.join(local_odpair_base_folder, "*.csv"))
# Read and combine all json files
od_pair_base_data = pd.concat([pd.read_csv(f) for f in od_pair_base_files], ignore_index=False)

print(f"Total number of od-pairs: {len(od_pair_base_data)}")
print(od_pair_base_data.columns)


# The od-pair data contains lists and dictionaries that are not easily saved to a csv file, so we store it as a json file.
# Still, there some columns that need to be serialized to strings such as shapely polygon objects.

od_pair_data_base_path_csv = os.path.join(local_odpair_folder, 'origin_od_pair_base.csv')

od_pair_base_data.to_csv(od_pair_data_base_path_csv)

In [ ]:
import pandas as pd # for reading the csv file
import joblib # replacing multiprocessing with joblib
import route_network_analysis as rna
import os
import csv
import glob


min_od_distance = 4750
max_od_distance = 5250
od_pair_sample_size = 10
experiment_name = "2025-07-evaluation"
base_path=f"/home/arvidh/Documents/GitHub/proj_full-analysis/evaluation/{experiment_name}"
city_sample_nodes_path = os.path.join(base_path, f'city_sample_nodes_{experiment_name}.csv')
df = pd.read_csv(city_sample_nodes_path)
#display(df)
local_odpair_folder = os.path.join(base_path, "od_pair_data")
local_odpair_base_folder = os.path.join(local_odpair_folder, 'base')
local_odpair_geom_folder = os.path.join(local_odpair_folder, 'geom')
print(f"odpair data will be stored at {local_odpair_folder}")
os.makedirs(local_odpair_folder, exist_ok=True)
os.makedirs(local_odpair_base_folder, exist_ok=True)
os.makedirs(local_odpair_geom_folder, exist_ok=True)

plotpath = "evaluation/2025-07-evaluation/plots"


# Get all json files from od_pair_data folder
od_pair_base_files = glob.glob(os.path.join(local_odpair_base_folder, "*.csv"))
# Read and combine all json files
od_pair_base_data = pd.concat([pd.read_csv(f) for f in od_pair_base_files], ignore_index=False)

print(f"Total number of od-pairs: {len(od_pair_base_data)}")
print(od_pair_base_data.columns)


# The od-pair data contains lists and dictionaries that are not easily saved to a csv file, so we store it as a json file.
# Still, there some columns that need to be serialized to strings such as shapely polygon objects.

od_pair_data_base_path_csv = os.path.join(local_odpair_folder, 'origin_od_pair_base.csv')

od_pair_base_data.to_csv(od_pair_data_base_path_csv)